# Experiment analysis
Run `python main.py` first so `mlflow.db` exists. This notebook reads the runs back with the MLflow API (the same data the UI shows).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import mlflow, pandas as pd, matplotlib.pyplot as plt
import config
from src import model_registry
mlflow.set_tracking_uri(config.TRACKING_URI)

## 1. All runs, best first (F1, then ROC-AUC)

In [2]:
table = model_registry.compare_runs()
table.drop(columns=['logged_model_uri']).round(4)

,run_id,model_name,accuracy,precision,recall,f1_score,roc_auc,training_time_seconds
0,9412c44f3f1549fe9c0d0a1dc1802a32,LogisticRegression,0.9825,0.9861,0.9861,0.9861,0.9954,0.0094
1,0f00c309bec84b1680734c2389c9850f,GradientBoosting,0.9561,0.9467,0.9861,0.9660,0.9931,0.4523
2,cbf28f66f2854f33b09a698581104ea9,RandomForest,0.9474,0.9583,0.9583,0.9583,0.9937,0.2455


## 2. Winner per metric

In [3]:
model_registry.summarise_winners(table)

{'accuracy': 'LogisticRegression',
 'f1_score': 'LogisticRegression',
 'roc_auc': 'LogisticRegression'}

## 3. Visual comparison

In [4]:
latest = table.drop_duplicates('model_name')
latest.set_index('model_name')[['accuracy','f1_score','roc_auc']].plot.bar(ylim=(0.9,1.0), figsize=(7,4), rot=0)
plt.title('Test-set metrics by model'); plt.show()

## 4. Best run and its registered version

In [5]:
best = model_registry.find_best_run()
print(best['model_name'], best['run_id'], round(best['f1_score'],4))

LogisticRegression 9412c44f3f1549fe9c0d0a1dc1802a32 0.9861


In [6]:
model = model_registry.load_registered_model(config.REGISTERED_MODEL_NAME, alias=config.MODEL_ALIAS)
from src.data_preprocessing import get_data
data = get_data()
model_registry.predict_with_registered_model(model, data.X_test.head(5))

array([0, 1, 0, 1, 0])